In [ ]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

load_dotenv()

In [ ]:
FILE_PATH = "./data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf"

In [ ]:
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import re

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()

cleaned = []
for doc in docs:
    t = doc.page_content
    t = re.sub(r'^\s*\d+\s*\n', '', t)                        # 문서 내 쪽번호 제거
    t = re.sub(r'^생성형 AI윤리 가이드북\s*\n', '', t, flags=re.M)  # 반복 표지 문구 제거
    t = re.sub(r'[ \t]+\n', '\n', t)                           # 줄 끝 공백 제거
    t = re.sub(r'\n{3,}', '\n\n', t)                           # 빈 줄 여러 개 → 하나로
    t = re.sub(r'[ \t]{2,}', ' ', t)                           # 연속 공백 → 한 칸
    doc.page_content = t.strip()
    # if doc.page_content:                                       # 빈 페이지(표지) 제외
    cleaned.append(doc)
docs = cleaned

print(len(docs))
print(docs[1].page_content[:200])

In [ ]:
show_metadata(docs)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
chunks = text_splitter.split_documents(docs)

print("원본 페이지 수:", len(docs))
print("Chunk 개수:", len(chunks))

In [ ]:
print(chunks[0].page_content)

In [ ]:
print(chunks[1].metadata)

In [ ]:
# #Embeddings
# #!pip install langchain-teddynote

# from langchain_teddynote import logging

# logging.langsmith("practice_rag")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 위에서 이미 만든 chunks(헤더 제거 + chunk_size=500 적용본)를 그대로 재사용해 FAISS 벡터 데이터베이스를 생성합니다.
db = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = db.as_retriever(search_kwargs={"k":3})

docs = retriever.invoke("대학교에서 생성형 AI 활용과 관련해 표절 문제를 예방하기 위한 절차나 조치를 사례를 들어 설명하세요")

for doc in docs:
    print(doc.page_content)
    print("=======================")

In [ ]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [ ]:
retriever = db.as_retriever(
    search_kwargs={"k": 3},
)
# 관련 문서를 검색
for doc in retriever.invoke("허위조작정보"):
    print(doc.page_content)
    print("=========================================================")

In [ ]:
#!pip install rank_bm25 kiwipiepy
#!pip install langchain-classic
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers import EnsembleRetriever   

kiwi = Kiwi()
def kiwi_tokenize(text):
    return [t.form for t in kiwi.tokenize(text)]   # "저작권은" → "저작권", "은"

bm25 = BM25Retriever.from_documents(chunks, preprocess_func=kiwi_tokenize)
bm25.k = 4

faiss_retriever = db.as_retriever(search_kwargs={"k": 4})

retriever = EnsembleRetriever(
    retrievers=[bm25, faiss_retriever],
    weights=[0.4, 0.6],
)

In [ ]:
#질문 확인용 코드
question = "생성형 AI가 무엇인가요?"

docs = retriever.invoke(question)

for i, doc in enumerate(docs, start=1):
    print(f"===== 검색 결과 {i} =====")
    print(doc.page_content)
    print()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 프롬프트를 생성
prompt = ChatPromptTemplate.from_template(
    """너는 '생성형 AI 윤리 가이드북' 안내 챗봇이다.

[규칙]
1. 반드시 아래 [문서 내용]에 있는 정보만 근거로 답한다. 너의 사전 지식은 쓰지 않는다.
2. 문서에 조건이나 예외(예: "~한 경우가 아니라면", "~라면")가 있으면 빠뜨리지 말고 함께 쓴다.
3. 숫자, 날짜, 기관명, 사례명은 문서에 적힌 그대로 쓴다.
4. 질문이 여러 가지를 묻고 있으면 각각 나눠서 답한다.
5. 문서에서 답을 찾을 수 없으면 "문서에서 확인할 수 없습니다"라고 답한다.
6. 답변 끝에 근거가 된 문서의 [출처: p.번호] 태그에 적힌 번호를 (p.번호) 형식으로 적는다. 본문 오른쪽 하단에 있는 숫자(인쇄된 쪽번호)는 페이지 번호로 사용하지 않는다.
7. 한국어로 답한다.
8. 자료를 바탕으로 유추하거나 상상해서 답변하지 말고, 문서에 직접적으로 언급된 사실만 사실로 인정하고 답변한다. 

[문서내용]: 
{docs}

[질문]:
{question}
"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
def format_docs(docs):
    return "\n\n".join(
        f"[출처: p.{doc.metadata['page'] + 1}]\n{doc.page_content}"
        for doc in docs
    )

chain = (
    {"docs": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
#질문없을때 답변

answer_chain = prompt | llm | StrOutputParser()   # retriever 없는 버전
history = []                                      # 나중에 RAGAS 평가용

def ask(question):
    found = retriever.invoke(question)                       # 1) 검색
    answer = answer_chain.invoke({                           # 2) 검색 결과로 답변
        "docs": format_docs(found),
        "question": question,
    })
    pages = sorted({d.metadata.get("page", 0) + 1 for d in found})

    print("답변:", answer)
    print("참고 페이지:", pages)

    history.append({                                         # 3) 기록
        "user_input": question,
        "response": answer,
        "retrieved_contexts": [d.page_content for d in found],
    })
    return answer

In [ ]:
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "생성형 AI가 무엇인가요?"
response = chain.invoke(question)
print(response)

In [ ]:
questions = [
    "생성형 AI가 제공하는 편리함에 따른 오남용 사례와 그로 인한 사회적 위험을 분석하고, 인간 고유의 능력과 AI 활용의 균형 필요성을 논하시오.",
    
    "대학교에서 생성형 AI 활용과 관련해 표절 문제를 예방하기 위한 절차나 조치를 사례를 들어 설명하세요.",
    
    "생성형 AI가 무엇이며, 어떤 방식으로 새로운 결과물을 만들어 내나요?",
    
    "웹툰 및 웹소설 플랫폼에서 생성형 AI가 제작한 콘텐츠를 이유로 환불을 요청하는 것이 법적 근거가 부족한 이유를 생성형 AI 윤리 가이드라인과 관련 사례를 통해 분석하시오.",

    "생성형 AI에 입력하는 내용이 외부로 유출될 위험이 있는 이유는 무엇인가요?",

    "생성형 AI가 인간의 편향된 데이터를 학습할 경우 발생할 수 있는 사회적 문제를 사례와 함께 설명하세요.",

    "생성형 AI에 과도하게 의존하는 것이 왜 문제인지, 그리고 인간 고유의 능력과 AI 활용의 올바른 관계를 설명하세요.",

    "생성형 AI를 이용해 가짜 뉴스를 제작·배포할 경우, 피해 의도가 없더라도 법적 책임이 발생할 수 있는 상황과 그 근거를 설명하세요.",

    "생성형 AI가 만든 정보의 진위 여부를 확인하기 위해 이용자가 가져야 할 습관은 무엇인가요?",

    
]

In [ ]:
import textwrap
for i, question in enumerate(questions, start=1):

    print("=" * 70)
    print(f"질문 {i}")
    print(question)
    print()

    answer = chain.invoke(question)

    print("답변:")
    print(textwrap.fill(answer, width=100))
    print()

In [ ]:
#!pip install datasets
from datasets import Dataset

data = []

for question in questions:
    
    # 검색된 문서
    retrieved_docs = retriever.invoke(question)
    
    # 문서 내용을 문자열로 변환
    contexts = [doc.page_content for doc in retrieved_docs]
    
    # RAG 답변
    answer = chain.invoke(question)
    
    data.append({
        "user_input": question,
        "response": answer,
        "retrieved_contexts": contexts
    })

dataset = Dataset.from_list(data)

dataset

In [ ]:
from openai import AsyncOpenAI
from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory

# ascore()가 비동기 평가 메서드이므로 AsyncOpenAI 클라이언트 사용
# API 응답을 기다리는 동안 노트북의 실행 흐름을 차단하지 않음
# 답변과 검색 문서를 판정할 평가용 LLM
evaluator_llm = llm_factory(
    model="gpt-4o-mini",
    client=AsyncOpenAI(),
    temperature=0,
    max_tokens=8192, 
)

# 비동기 평가에서 의미 유사도를 계산할 임베딩 모델
evaluator_embeddings = OpenAIEmbeddings(
    client=AsyncOpenAI(),
    model="text-embedding-3-small",
)

In [ ]:
from ragas.metrics.collections import AnswerRelevancy, Faithfulness

# reference(정답) 데이터가 없으므로, reference 없이 계산 가능한
# Faithfulness / AnswerRelevancy만 dataset 전체(10문항)에 대해 평가합니다.
# (ContextPrecisionWithReference, ContextRecall은 reference가 필요해 제외)
eval_results = []

for sample in dataset:
    # 생성 평가 1: 답변의 내용이 검색 문서에 근거하는지 확인
    faithfulness = await Faithfulness(llm=evaluator_llm).ascore(
        user_input=sample["user_input"],
        response=sample["response"],
        retrieved_contexts=sample["retrieved_contexts"],
    )

    # 생성 평가 2: 답변이 질문에 직접 관련되는지 확인
    answer_relevancy = await AnswerRelevancy(
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        strictness=1,
    ).ascore(
        user_input=sample["user_input"],
        response=sample["response"],
    )

    eval_results.append({
        "user_input": sample["user_input"],
        "faithfulness": faithfulness.value,
        "answer_relevancy": answer_relevancy.value,
    })

for r in eval_results:
    print(f"[faithfulness={r['faithfulness']:.2f}, answer_relevancy={r['answer_relevancy']:.2f}] {r['user_input']}")